In [1]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, RepeatVector, Dense
from sklearn.preprocessing import StandardScaler

# Load sensor data and keep healthy training examples
df = pd.read_csv("LP IV Updated Datasets/AutoEncoder/NASA Turbofan ENgine/NASA TurboFan Engine.csv")
features = df.filter(like="sensor").columns
train = df[(df.unit <= 80) & (df.Fault_Label == 0)]
test = df[df.unit > 80]

scaler = StandardScaler()
scaler.fit(train[features])
window_size = 20

# Make sequences of 20 readings from each engine
def create_sequences(data):
    sequences = []
    for engine_id, engine in data.groupby("unit"):
        engine = engine.sort_values("cycle")
        readings = scaler.transform(engine[features])
        for start in range(len(readings) - window_size + 1):
            end = start + window_size
            sequence = readings[start:end]
            sequences.append(sequence)
    return np.array(sequences)

x_train = create_sequences(train)
x_test = create_sequences(test)

# Create LSTM Autoencoder
model = Sequential([
    Input(shape=(window_size, len(features))),
    LSTM(16),
    RepeatVector(window_size),
    LSTM(16, return_sequences=True),
    Dense(len(features))
])
model.compile(optimizer="adam", loss="mse")
model.fit(x_train, x_train, epochs=10, batch_size=128)

# Set a threshold using healthy training errors
train_prediction = model.predict(x_train)
train_error = np.mean((x_train - train_prediction) ** 2, axis=(1, 2))
threshold = np.percentile(train_error, 95)

# Detect unusual test sequences
test_prediction = model.predict(x_test)
test_error = np.mean((x_test - test_prediction) ** 2, axis=(1, 2))
faults = test_error > threshold
print("Faults:", faults.sum())


Epoch 1/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 0.5134
Epoch 2/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.3385
Epoch 3/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.3033
Epoch 4/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.2933
Epoch 5/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.2886
Epoch 6/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.2859
Epoch 7/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.2840
Epoch 8/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.2827
Epoch 9/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.2817
Epoch 10/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.2809
380/380 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Faults: 251
